In [2]:
import os
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt 

def process_image(input_image_path, output_image_path):
    # Read the image
    image = cv2.imread(input_image_path, cv2.IMREAD_GRAYSCALE)

    # Define thresholds for light and dark grey pixels
    light_threshold = 200
    dark_threshold = 100

    # Create masks for light and dark grey pixels
    light_mask = (image > light_threshold)
    dark_mask = (image <= dark_threshold)

    # Set light grey pixels to white and dark grey pixels to black
    image[light_mask] = 255
    image[dark_mask] = 0

    # Save the processed image
    cv2.imwrite(output_image_path, image)

def crop_image(image_path, crop_size):
    """
    Crop the image into smaller parts.

    Parameters:
        image_path (str): Path to the image file.
        crop_size (tuple): Size of each cropped part (width, height).

    Returns:
        list: List of cropped images.
    """
    # Open the image
    image = Image.open(image_path)
    
    # Get the dimensions of the original image
    width, height = image.size
    
    # Calculate the number of rows and columns of cropped parts
    num_rows = height // crop_size[1]
    num_cols = width // crop_size[0]
    
    # List to store the cropped images
    cropped_images = []
    
    # Crop the image into smaller parts
    for y in range(num_rows):
        for x in range(num_cols):
            left = x * crop_size[0]
            upper = y * crop_size[1]
            right = left + crop_size[0]
            lower = upper + crop_size[1]
            
            # Crop the part of the image
            cropped_part = image.crop((left, upper, right, lower))
            
            # Append the cropped part to the list
            cropped_images.append(cropped_part)
    
    return cropped_images

def get_bounding_boxes(image_path):
    """
    Get bounding boxes around each white/grey part in the image.

    Parameters:
        image_path (str): Path to the image file.

    Returns:
        list: List of bounding boxes ((x, y, w, h)) around each white/grey part.
    """
    # Load the image
    image = cv2.imread(image_path)

    # Convert the image to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Apply binary thresholding to separate white/grey parts from black background
    _, binary = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)

    # Find contours of white/grey parts
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Get bounding boxes around contours
    bounding_boxes = [cv2.boundingRect(cnt) for cnt in contours]

    return bounding_boxes



def stitch_images(image_paths, num_cols):
    """
    Stitch cropped images back together into a single image.

    Parameters:
        image_paths (list): List of paths to the cropped image files.
        num_cols (int): Number of columns in the grid of cropped images.

    Returns:
        PIL.Image: The stitched image.
    """
    # Open the cropped images
    cropped_images = [Image.open(path) for path in image_paths]

    # Get the dimensions of the first cropped image
    width, height = cropped_images[0].size

    # Calculate the number of rows in the grid of cropped images
    num_rows = len(cropped_images) // num_cols

    # Create a new blank image to stitch the cropped images onto
    stitched_image = Image.new('RGB', (width * num_cols, height * num_rows))

    # Iterate over the cropped images and paste them onto the stitched image
    for i, img in enumerate(cropped_images):
        col_index = i % num_cols
        row_index = i // num_cols
        stitched_image.paste(img, (col_index * width, row_index * height))

    return stitched_image
def merge_bounding_boxes_on_stitched(bounding_boxes, stitched_image_width, stitched_image_height, threshold_distance):
    """
    Merge bounding boxes that are close to each other on the stitched image.

    Parameters:
        bounding_boxes (list): List of bounding boxes ((x, y, w, h)).
        stitched_image_width (int): Width of the stitched image.
        stitched_image_height (int): Height of the stitched image.
        threshold_distance (int): Maximum distance to consider bounding boxes as close.

    Returns:
        list: List of merged bounding boxes ((x, y, w, h)).
    """
    merged_boxes = []
    for box in bounding_boxes:
        x, y, w, h = box
        merged = False
        for i, merged_box in enumerate(merged_boxes):
            x_merge, y_merge, w_merge, h_merge = merged_box
            # Calculate distance between the centers of bounding boxes
            center_x = x + w // 2
            center_y = y + h // 2
            center_x_merge = x_merge + w_merge // 2
            center_y_merge = y_merge + h_merge // 2
            distance = np.sqrt((center_x - center_x_merge)**2 + (center_y - center_y_merge)**2)
            # If the distance is less than the threshold, merge the bounding boxes
            if distance < threshold_distance:
                x_new = min(x, x_merge)
                y_new = min(y, y_merge)
                w_new = max(x + w, x_merge + w_merge) - x_new
                h_new = max(y + h, y_merge + h_merge) - y_new
                # Ensure the merged bounding box stays within the stitched image dimensions
                x_new = max(0, x_new)
                y_new = max(0, y_new)
                w_new = min(w_new, stitched_image_width - x_new)
                h_new = min(h_new, stitched_image_height - y_new)
                merged_boxes[i] = (x_new, y_new, w_new, h_new)
                merged = True
                break
        if not merged:
            merged_boxes.append(box)
    return merged_boxes
            

def stitch_images_with_bounding_boxes(images_dir, output_dir, crop_size, num_cols):
    """
    Stitch cropped images with bounding boxes and save them to the output directory.

    Parameters:
        images_dir (str): Directory containing input images.
        output_dir (str): Directory to save the stitched images.
        crop_size (tuple): Size of each cropped part (width, height).
        num_cols (int): Number of columns in the grid of cropped images.
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Get list of image files in the directory
    image_files = [f for f in os.listdir(images_dir) if f.endswith('.tif')]

    # Process each image
    for image_file in image_files:
        # Path to input and output images
        input_image_path = os.path.join(images_dir, image_file)
        output_image_path = os.path.join(output_dir, image_file)

        # Process the image
        process_image(input_image_path, output_image_path)

        # Crop the processed image
        cropped_images = crop_image(output_image_path, crop_size)
        cropped_output_image_paths = []
        all_bounding_boxes = []

        for i, img in enumerate(cropped_images):
            cropped_path = os.path.join(output_dir, 'cropped')
            os.makedirs(cropped_path, exist_ok=True)
            cropped_output_image_path = os.path.join(cropped_path, f'cropped_output_image_{i}.jpg')
            cropped_output_image_paths.append(cropped_output_image_path)
            img.save(cropped_output_image_path)

            # Get bounding boxes for each cropped image
            bounding_boxes = get_bounding_boxes(cropped_output_image_path)

            # Rescale bounding boxes to match the stitched image coordinates
            row_index = i // num_cols
            col_index = i % num_cols
            rescaled_bounding_boxes = [(box[0] + col_index * crop_size[0],
                                        box[1] + row_index * crop_size[1],
                                        box[2], box[3]) for box in bounding_boxes]

            # Append rescaled bounding boxes to the list
            all_bounding_boxes.extend(rescaled_bounding_boxes)

        # Stitch the cropped images back together into a single image
        stitched_image = stitch_images(cropped_output_image_paths, num_cols)
        stitched_image=Image.open(input_image_path)
        
        # Get the width and height of the stitched image
        stitched_image_width, stitched_image_height = stitched_image.size

        # Merge bounding boxes on the stitched image
        merged_bounding_boxes = merge_bounding_boxes_on_stitched(all_bounding_boxes,
                                                                  stitched_image_width,
                                                                  stitched_image_height,
                                                                  threshold_distance=0.01)

        # Draw merged bounding boxes on the stitched image
        stitched_image_np = np.array(stitched_image)
        for box in merged_bounding_boxes:
            x, y, w, h = box
            cv2.rectangle(stitched_image_np, (x, y), (x + w, y + h), (0, 255, 0), 1)

        # Save the stitched image with merged bounding boxes
        stitched_image_with_bbs = Image.fromarray(stitched_image_np)
        stitched_image_with_bbs.save(os.path.join(output_dir, f'stitched_{image_file}'))
       

def coal_segmentation(images_dir, mineral_seg_dir):


    
    # Get list of image files in the directory
    image_files = [f for f in os.listdir(images_dir) if f.endswith('.tif')]

    # Process each image
    for image_file in image_files:
        # Path to input and output images
        input_image_path = os.path.join(images_dir, image_file)
        mineral_seg_path= os.path.join(mineral_seg_dir, 'stitched_'+image_file)
        
        # Define thresholds for coal patches
        coal_lower_threshold = 20
        coal_upper_threshold =60

        min_coal_patch_area = 500 # Adjust this value as needed
        
            
        image = cv2.imread(input_image_path, cv2.IMREAD_GRAYSCALE)
        # Calculate the current average brightness
        average_brightness = cv2.mean(image)[0]

        # Define the target average brightness and the acceptable range
        target_brightness = 20

        # Calculate the brightness factor
        brightness_factor = (target_brightness - average_brightness) / average_brightness

        # Define the brightness reduction factor (should be less than 1)
        if brightness_factor<0:
            brightness_factor = 1-abs(brightness_factor)# Example: Reducing brightness by 50%



        # Multiply each pixel value by the brightness factor
        image = np.clip(image * brightness_factor, 0, 255).astype(np.uint8)
        # Create a mask for coal patches
        coal_mask = cv2.inRange(image, coal_lower_threshold, coal_upper_threshold)

        # Clean up the mask (optional)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        coal_mask = cv2.morphologyEx(coal_mask, cv2.MORPH_OPEN, kernel)
        coal_mask = cv2.morphologyEx(coal_mask, cv2.MORPH_CLOSE, kernel)

        # Find contours of coal patches
        contours, _ = cv2.findContours(coal_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)


        image = cv2.imread(mineral_seg_path)

        # Create a copy of the original image for visualization
        coal_image = image.copy()

        # Draw bounding boxes around coal patches
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area >= min_coal_patch_area:
                x, y, w, h = cv2.boundingRect(cnt)
                cv2.rectangle(coal_image, (x, y), (x + w, y + h), (255, 0, 0), 2)
    
        plt.imsave(mineral_seg_path.split('.')[0]+'_final.jpg',coal_image)

# Example usage
images_dir = 'original_coal_images'  # Input directory containing images
output_dir = 'output_images'  # Output directory to save stitched images
crop_size = (200, 200)  # Size of each cropped part
num_cols = 10  # Number of columns in the grid of cropped images
stitch_images_with_bounding_boxes(images_dir, output_dir, crop_size, num_cols)
coal_segmentation(images_dir,output_dir)
